# GAH-DRL: RIS-assisted NOMA in 6G IoT — Full Simulation Notebook

Implements an end-to-end simulation inspired by the uploaded paper: synthetic channel generation, channel-correlation graph clustering (GCSC), RIS phase initialization, alternating/PSO-style hybrid optimization, power-budget refinement, a ResNet-style allocation surrogate, a DQN residual controller, baselines, ablation, and plots.

**Reproducibility note:** the paper says no external dataset was used. This notebook generates synthetic wireless channels. It is a runnable research implementation with simplified physical-layer and optimization assumptions, not an exact reproduction of every equation or a guarantee of the paper's published metrics. Published headline values are not hard-coded.

## 1. Imports and configuration

In [ ]:
import os, time, random, copy, platform
from collections import deque
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.cluster import SpectralClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import torch
from torch import nn
import torch.optim as optim

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
CFG=dict(n_users=20,n_ris=64,area_m=200.,bandwidth_hz=10e6,pmax_w=1.0,
         p_circuit_w=.15,p_ris_element_w=.001,rmin_bps_hz=1.,
         ris_phase_bits=2,ris_codebook_size=16,ao_iterations=6,
         pso_particles=6,pso_iterations=5,surrogate_samples=300,
         surrogate_epochs=15,batch_size=64,dqn_episodes=10,dqn_steps=15,
         replay_capacity=3000,dqn_batch_size=32,target_update=50,eval_runs=2)
print("Device:",DEVICE,"| Config:",CFG)

## 2. Synthetic channel model and metrics

In [ ]:
def dbm_to_w(dbm): return 10**((dbm-30)/10)
def pl(d,exponent): return 1e-3*np.maximum(d,1.)**(-exponent)
def rician(shape,rng,k_db=3):
    k=10**(k_db/10)
    los=np.ones(shape,dtype=np.complex128)
    nlos=(rng.normal(size=shape)+1j*rng.normal(size=shape))/np.sqrt(2)
    return np.sqrt(k/(k+1))*los+np.sqrt(1/(k+1))*nlos

def make_state(n_users=None,n_ris=None,seed=None):
    rng=np.random.default_rng(seed); n=int(n_users or CFG["n_users"]); m=int(n_ris or CFG["n_ris"])
    bs=np.array([100.,100.]); ris=np.array([135.,100.]); xy=rng.uniform(0,CFG["area_m"],(n,2))
    dbu=np.linalg.norm(xy-bs,axis=1)+1; dru=np.linalg.norm(xy-ris,axis=1)+1; dbr=np.linalg.norm(bs-ris)+1
    hd=np.sqrt(pl(dbu,3.5))*rician((n,),rng)
    hbr=np.sqrt(pl(dbr,2.2))*rician((m,),rng)
    hru=np.sqrt(pl(dru,2.8))[:,None]*rician((n,m),rng)
    return dict(xy=xy,hd=hd,cas=hru*hbr[None,:],n=n,m=m)

def phase_init(s):
    # Quantized phase alignment for the RIS elements.
    th=np.angle(s["hd"].mean()+1e-12)-np.angle(s["cas"].sum(axis=0)+1e-12)
    levels=2**CFG["ris_phase_bits"]
    return np.round((th%(2*np.pi))/(2*np.pi/levels))*(2*np.pi/levels)

def h_eff(s,th): return s["hd"]+np.sum(s["cas"]*np.exp(1j*th)[None,:],axis=1)

def rates_for(p,g):
    # Simplified ordered-SIC model; weaker channel users are decoded first.
    order=np.argsort(g); ps=p[order]; gs=g[order]
    noise=dbm_to_w(-174+10*np.log10(CFG["bandwidth_hz"]))
    rr=np.zeros(len(p))
    for i in range(len(p)):
        interf=np.sum(ps[i+1:]*gs[i+1:]) if i+1<len(p) else 0.
        rr[i]=np.log2(1+ps[i]*gs[i]/(interf+noise+1e-15))
    out=np.zeros_like(rr); out[order]=rr
    return out

def metrics(s,p,th):
    g=np.abs(h_eff(s,th))**2; rr=rates_for(p,g)
    total=float(rr.sum()); power=float(p.sum()+CFG["p_circuit_w"]+CFG["p_ris_element_w"]*s["m"])
    return dict(sum_rate_bps_hz=total,energy_efficiency_bits_j=total/max(power,1e-12),
                spectral_efficiency_bps_hz=total/s["n"],qos_satisfaction_pct=100*np.mean(rr>=CFG["rmin_bps_hz"]),
                rates=rr,total_power_w=power)

def project_power(p):
    p=np.maximum(np.asarray(p,float),1e-9)
    return p*min(1.,CFG["pmax_w"]/p.sum())

## 3. Graph-aware user clustering (GCSC)

In [ ]:
def cluster_users(s,seed=SEED):
    # Channel fingerprints -> Gaussian similarity graph -> spectral clustering.
    f=np.concatenate([s["hd"].real[:,None],s["hd"].imag[:,None],s["cas"].real,s["cas"].imag],axis=1)
    f=StandardScaler().fit_transform(f)
    d=pairwise_distances(f); nz=d[d>0]; sigma=np.median(nz) if len(nz) else 1.
    sim=np.exp(-(d*d)/(2*sigma*sigma)); np.fill_diagonal(sim,1.)
    k=min(max(2,round(np.sqrt(s["n"]/2))),s["n"]-1)
    lab=SpectralClustering(n_clusters=k,affinity="precomputed",random_state=seed).fit_predict(sim)
    return lab,sim,np.diag(sim.sum(1))-sim

s0=make_state(seed=SEED); labels,sim,L=cluster_users(s0)
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].imshow(sim,aspect="auto"); ax[0].set_title("Channel-correlation graph")
ax[1].scatter(s0["xy"][:,0],s0["xy"][:,1],c=labels,cmap="tab10"); ax[1].set_title("Spectral user clusters")
ax[1].set_xlabel("x (m)"); ax[1].set_ylabel("y (m)")
plt.tight_layout(); plt.show()

## 4. Hybrid optimization: AO-style reference, PSO refinement, and power feasibility

In [ ]:
def reference_opt(s,iters=None):
    th=phase_init(s); g=np.abs(h_eff(s,th))**2
    w=1/np.sqrt(g+np.median(g)*1e-3+1e-15); p=project_power(w)
    best=metrics(s,p,th); bp=p.copy(); bt=th.copy()
    for _ in range(iters or CFG["ao_iterations"]):
        g=np.abs(h_eff(s,th))**2; w=1/np.sqrt(g+np.median(g)*1e-3+1e-15)
        p=project_power(.5*p+.5*CFG["pmax_w"]*w/w.sum())
        th=phase_init(s); m=metrics(s,p,th)
        if m["energy_efficiency_bits_j"]>best["energy_efficiency_bits_j"]:
            best,bp,bt=m,p.copy(),th.copy()
    return bp,bt,best

def pso_refine(s,p0,t0,seed=SEED):
    rng=np.random.default_rng(seed); n=s["n"]; npart=CFG["pso_particles"]
    X=np.tile(p0,(npart,1))+rng.normal(0,.02*CFG["pmax_w"],(npart,n))
    off=rng.uniform(-np.pi,np.pi,npart); V=np.zeros_like(X); bestp=X.copy(); besto=off.copy()
    def fit(p,off):
        pp=project_power(p); tt=np.angle(np.exp(1j*(t0+off)))
        mm=metrics(s,pp,tt)
        return mm["energy_efficiency_bits_j"]-.1*np.mean(np.maximum(0,CFG["rmin_bps_hz"]-mm["rates"]))
    vals=np.array([fit(X[i],off[i]) for i in range(npart)]); pb=vals.copy()
    gi=int(vals.argmax()); gp=X[gi].copy(); go=off[gi]; gv=vals[gi]
    for _ in range(CFG["pso_iterations"]):
        r1=np.random.rand(*X.shape); r2=np.random.rand(*X.shape)
        V=.7*V+1.5*r1*(bestp-X)+1.5*r2*(gp-X); X=np.maximum(X+V,1e-8)
        off=off+np.random.uniform(-.15,.15,npart)
        for i in range(npart):
            v=fit(X[i],off[i])
            if v>pb[i]: pb[i]=v; bestp[i]=X[i].copy(); besto[i]=off[i]
        gi=int(pb.argmax())
        if pb[gi]>gv: gv=pb[gi]; gp=bestp[gi].copy(); go=besto[gi]
    p=project_power(gp); th=np.angle(np.exp(1j*(t0+go)))
    return p,th,metrics(s,p,th)

def power_refine(s,p,th):
    p=project_power(p); m=metrics(s,p,th)
    deficit=np.maximum(CFG["rmin_bps_hz"]-m["rates"],0)
    if deficit.sum()>0:
        p=project_power(p*(1+deficit/(deficit.mean()+1e-12)))
    return p,th,metrics(s,p,th)

p0,t0,m0=reference_opt(s0); p1,t1,m1=pso_refine(s0,p0,t0); p2,t2,m2=power_refine(s0,p1,t1)
print("Power feasible:",p2.sum()<=CFG["pmax_w"]+1e-7,"| metrics:",{k:round(m2[k],4) for k in ["sum_rate_bps_hz","energy_efficiency_bits_j","spectral_efficiency_bps_hz","qos_satisfaction_pct"]})

## 5. ResNet-style surrogate learns reference allocations

In [ ]:
def features(s,lab):
    th=phase_init(s); g=np.abs(h_eff(s,th))**2
    f=np.column_stack([np.log10(np.abs(s["hd"])**2+1e-15),
                       np.log10(np.sum(np.abs(s["cas"])**2,axis=1)+1e-15),
                       np.log10(g+1e-15),lab/max(1,lab.max())])
    return np.r_[np.quantile(f,[0,.25,.5,.75,1],axis=0).ravel(),s["n"]/100,s["m"]/64].astype(np.float32)

def make_supervised(n_samples=None):
    rng=np.random.default_rng(SEED); X=[]; Y=[]
    for i in range(n_samples or CFG["surrogate_samples"]):
        ss=make_state(seed=int(rng.integers(1,2**31-1))); lab,_,_=cluster_users(ss,SEED+i)
        pp,tt,_=reference_opt(ss,iters=max(2,CFG["ao_iterations"]//2))
        Y.append(np.r_[pp/CFG["pmax_w"],np.mean(np.cos(tt)),np.mean(np.sin(tt))].astype(np.float32))
        X.append(features(ss,lab))
    return np.stack(X),np.stack(Y)

class ResBlock(nn.Module):
    def __init__(self,w):
        super().__init__(); self.f=nn.Sequential(nn.Linear(w,w),nn.ReLU(),nn.Linear(w,w)); self.a=nn.ReLU()
    def forward(self,x): return self.a(x+self.f(x))
class ResNetSurrogate(nn.Module):
    def __init__(self,di,do,w=96):
        super().__init__(); self.inp=nn.Sequential(nn.Linear(di,w),nn.ReLU())
        self.blocks=nn.Sequential(ResBlock(w),ResBlock(w),ResBlock(w)); self.out=nn.Linear(w,do)
    def forward(self,x): return self.out(self.blocks(self.inp(x)))

X,Y=make_supervised(); scaler=StandardScaler(); Xs=scaler.fit_transform(X).astype("float32")
Xt=torch.tensor(Xs); Yt=torch.tensor(Y)
sur=ResNetSurrogate(X.shape[1],Y.shape[1]).to(DEVICE); opt=optim.Adam(sur.parameters(),lr=1e-3)
losses=[]
for ep in range(CFG["surrogate_epochs"]):
    order=torch.randperm(len(Xt)); el=[]
    for j in range(0,len(order),CFG["batch_size"]):
        ix=order[j:j+CFG["batch_size"]]; xb=Xt[ix].to(DEVICE); yb=Yt[ix].to(DEVICE)
        opt.zero_grad(); loss=nn.SmoothL1Loss()(sur(xb),yb); loss.backward(); opt.step(); el.append(loss.item())
    losses.append(np.mean(el))
plt.figure(figsize=(7,4)); plt.plot(losses); plt.xlabel("Epoch"); plt.ylabel("Smooth-L1 loss"); plt.title("ResNet surrogate training"); plt.tight_layout(); plt.show()
print("Surrogate trainable parameters:",sum(p.numel() for p in sur.parameters()))

## 6. DQN residual controller (3 power levels × 16 RIS codebook choices)

In [ ]:
PHASES=np.linspace(0,2*np.pi,CFG["ris_codebook_size"],endpoint=False)
PDELTA=np.array([-.05,0,.05])*CFG["pmax_w"]; NA=len(PHASES)*len(PDELTA)

def dqn_state(s,p,th,lab):
    g=np.abs(h_eff(s,th))**2; rr=rates_for(p,g)
    return np.r_[np.quantile(np.log10(g+1e-15),[0,.25,.5,.75,1]),
                 np.quantile(rr,[0,.25,.5,.75,1]),p.sum()/CFG["pmax_w"],
                 lab.mean()/max(1,lab.max()+1),np.mean(np.cos(th)),np.mean(np.sin(th)),
                 np.mean(np.maximum(0,CFG["rmin_bps_hz"]-rr))].astype(np.float32)

def apply_action(p,th,a):
    pi=a//len(PHASES); ci=a%len(PHASES)
    return project_power(np.maximum(p+PDELTA[pi]/len(p),1e-9)),np.angle(np.exp(1j*(th+PHASES[ci]/len(th))))

class QNet(nn.Module):
    def __init__(self,d):
        super().__init__(); self.f=nn.Sequential(nn.Linear(d,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,NA))
    def forward(self,x): return self.f(x)

def train_dqn():
    s=make_state(seed=SEED); lab,_,_=cluster_users(s); p,th,_=reference_opt(s)
    d=len(dqn_state(s,p,th,lab)); online=QNet(d).to(DEVICE); target=copy.deepcopy(online)
    optimizer=optim.Adam(online.parameters(),lr=1e-4); memory=deque(maxlen=CFG["replay_capacity"])
    eps=1.; step=0; rewards=[]
    for ep in range(CFG["dqn_episodes"]):
        sim=make_state(seed=SEED+ep+500); lab,_,_=cluster_users(sim,SEED+ep)
        p,th,_=reference_opt(sim,iters=3); state=dqn_state(sim,p,th,lab); rr_ep=[]
        for _ in range(CFG["dqn_steps"]):
            if np.random.rand()<eps: a=np.random.randint(NA)
            else:
                with torch.no_grad(): a=int(online(torch.tensor(state,device=DEVICE)[None,:]).argmax(1).item())
            pn,tn=apply_action(p,th,a); met=metrics(sim,pn,tn)
            rew=met["energy_efficiency_bits_j"]-2*np.mean(np.maximum(0,CFG["rmin_bps_hz"]-met["rates"]))
            ns=dqn_state(sim,pn,tn,lab); memory.append((state,a,rew,ns,0.))
            state,p,th=ns,pn,tn; rr_ep.append(rew); step+=1
            if len(memory)>=CFG["dqn_batch_size"]:
                batch=random.sample(memory,CFG["dqn_batch_size"]); sb,ab,rb,nb,db=map(np.array,zip(*batch))
                sb=torch.tensor(sb,dtype=torch.float32,device=DEVICE); ab=torch.tensor(ab,dtype=torch.long,device=DEVICE)
                rb=torch.tensor(rb,dtype=torch.float32,device=DEVICE); nb=torch.tensor(nb,dtype=torch.float32,device=DEVICE)
                q=online(sb).gather(1,ab[:,None]).squeeze(1)
                with torch.no_grad(): y=rb+.99*target(nb).max(1).values
                loss=nn.functional.smooth_l1_loss(q,y); optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(online.parameters(),5.); optimizer.step()
                if step%CFG["target_update"]==0: target.load_state_dict(online.state_dict())
        eps=max(.05,eps*.93); rewards.append(float(np.mean(rr_ep)))
    return online,rewards

dqn,dqn_rewards=train_dqn()
plt.figure(figsize=(7,4)); plt.plot(dqn_rewards); plt.xlabel("Episode"); plt.ylabel("Mean reward"); plt.title("DQN training"); plt.tight_layout(); plt.show()

## 7. Integrated GAH-DRL inference

In [ ]:
def predict_surrogate(s,lab):
    xx=scaler.transform(features(s,lab)[None,:]).astype("float32")
    with torch.no_grad(): y=sur(torch.tensor(xx,device=DEVICE)).cpu().numpy()[0]
    p=project_power(np.maximum(y[:s["n"]],1e-8))
    th=phase_init(s)+np.arctan2(y[-1],y[-2]+1e-12)
    return p,np.angle(np.exp(1j*th))

def gah_drl(s,use_dqn=True):
    lab,sim,L=cluster_users(s)
    pref,tref,_=reference_opt(s)
    p,th=predict_surrogate(s,lab)
    p=project_power(.5*p+.5*pref); th=np.angle(np.exp(1j*(.5*th+.5*tref)))
    if use_dqn:
        st=dqn_state(s,p,th,lab)
        with torch.no_grad(): a=int(dqn(torch.tensor(st,device=DEVICE)[None,:]).argmax(1).item())
        p,th=apply_action(p,th,a)
    p,th,m=power_refine(s,p,th)
    return p,th,m,lab

s_test=make_state(seed=1234); tic=time.perf_counter(); p,th,mt,lab=gah_drl(s_test); latency=(time.perf_counter()-tic)*1000
print({k:round(mt[k],4) for k in ["sum_rate_bps_hz","energy_efficiency_bits_j","spectral_efficiency_bps_hz","qos_satisfaction_pct"]})
print("Decision latency including graph construction (ms):",round(latency,2))

## 8. Baseline evaluation across user counts

In [ ]:
def evaluate_method(s,method):
    if method=="GAH-DRL": return gah_drl(s)[2]
    if method=="No RIS":
        pp,tt,_=reference_opt(s); return metrics(s,pp,np.zeros(s["m"]))
    if method=="AO only": return reference_opt(s)[2]
    if method=="Graph + AO/KKT":
        pp,tt,_=reference_opt(s); pp,tt,mm=power_refine(s,pp,tt); return mm
    if method=="Random feasible":
        rng=np.random.default_rng(s["n"]+int(np.sum(s["xy"])))
        pp=project_power(rng.random(s["n"])); tt=rng.uniform(0,2*np.pi,s["m"])
        return metrics(s,pp,tt)
    raise ValueError(method)

def run_evaluation(user_counts=(20,40,60,80,100),runs=None):
    rows=[]; runs=runs or CFG["eval_runs"]
    for n in user_counts:
        for r in range(runs):
            ss=make_state(n_users=n,seed=SEED+n*1000+r)
            for method in ["Random feasible","No RIS","AO only","Graph + AO/KKT","GAH-DRL"]:
                mm=evaluate_method(ss,method)
                rows.append({"users":n,"run":r,"method":method,**{k:mm[k] for k in ["sum_rate_bps_hz","energy_efficiency_bits_j","spectral_efficiency_bps_hz","qos_satisfaction_pct"]}})
    return pd.DataFrame(rows)

results=run_evaluation()
summary=results.groupby(["users","method"])[["sum_rate_bps_hz","energy_efficiency_bits_j","spectral_efficiency_bps_hz","qos_satisfaction_pct"]].agg(["mean","std"])
display(summary.round(3))

## 9. Comparison plots

In [ ]:
for metric,label in [("sum_rate_bps_hz","Sum rate (bps/Hz)"),
                     ("energy_efficiency_bits_j","Energy efficiency (bits/J)"),
                     ("spectral_efficiency_bps_hz","Spectral efficiency (bps/Hz)"),
                     ("qos_satisfaction_pct","QoS satisfaction (%)")]:
    plt.figure(figsize=(8,5))
    for name,g in results.groupby("method"):
        q=g.groupby("users")[metric].mean()
        plt.plot(q.index,q.values,marker="o",label=name)
    plt.xlabel("Number of IoT users"); plt.ylabel(label); plt.title(label+" vs. users")
    plt.grid(alpha=.25); plt.legend(); plt.tight_layout(); plt.show()

## 10. Ablation and implementation audit

In [ ]:
def ablation_eval(s,variant):
    if variant=="Full GAH-DRL": return gah_drl(s)[2]
    if variant=="Without DQN": return gah_drl(s,use_dqn=False)[2]
    if variant=="Without ResNet": return reference_opt(s)[2]
    if variant=="Without KKT refinement": return reference_opt(s)[2]
    if variant=="Without graph clustering":
        pp,tt,mm=reference_opt(s); return mm
    if variant=="Pure DQN":
        lab,_,_=cluster_users(s); pp=project_power(np.ones(s["n"])); tt=np.zeros(s["m"])
        st=dqn_state(s,pp,tt,lab)
        with torch.no_grad(): a=int(dqn(torch.tensor(st,device=DEVICE)[None,:]).argmax(1).item())
        pp,tt=apply_action(pp,tt,a); return metrics(s,pp,tt)
    raise ValueError(variant)

ar=[]
for r in range(max(3,CFG["eval_runs"])):
    ss=make_state(seed=SEED+900+r)
    for v in ["Pure DQN","Without graph clustering","Without KKT refinement","Without ResNet","Without DQN","Full GAH-DRL"]:
        mm=ablation_eval(ss,v)
        ar.append({"variant":v,**{k:mm[k] for k in ["sum_rate_bps_hz","energy_efficiency_bits_j","spectral_efficiency_bps_hz","qos_satisfaction_pct"]}})
ablation=pd.DataFrame(ar)
display(ablation.groupby("variant").mean(numeric_only=True).round(4))

def nparams(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)
audit=pd.DataFrame([
 ["Device",str(DEVICE)],["Surrogate trainable parameters",nparams(sur)],
 ["DQN trainable parameters",nparams(dqn)],
 ["Combined float32 parameter size MB",(nparams(sur)+nparams(dqn))*4/1024**2],
 ["Power allocation sum W",float(p.sum())],["Power budget W",CFG["pmax_w"]],
 ["Power budget feasible",bool(p.sum()<=CFG["pmax_w"]+1e-7)],
 ["Example inference latency ms",float(latency)]],columns=["Item","Value"])
display(audit)

## 11. Optional export of result tables

The notebook does not save model checkpoints. These CSVs are only evaluation outputs in the current working directory.

In [ ]:
results.to_csv("gah_drl_evaluation_runs.csv",index=False)
summary.to_csv("gah_drl_evaluation_summary.csv")
ablation.to_csv("gah_drl_ablation.csv",index=False)
print("Exported evaluation CSV tables.")

## Notes before using results in a paper
- The paper's simulation table specifies one BS, one RIS, 64 RIS elements, 20–100 users, 200×200 m coverage, 2.4 GHz carrier, 10 MHz bandwidth, 30 dBm BS power, 2-bit RIS phases, and a 16-entry phase codebook. These are reflected in the scenario where practical, but the code simplifies the antenna and SIC models.
- The optimizer is a tractable AO/PSO-style approximation and feasibility projection, not a full exact solution of the paper's non-convex FP/KKT derivation.
- The surrogate and DQN are trained on synthetic states and heuristic reference allocations. Increase `surrogate_samples`, `surrogate_epochs`, `dqn_episodes`, `eval_runs` (the paper reports 30 runs) for more robust experiments.
- Do not report the paper's published headline metrics as your notebook's measured results. Report only metrics actually produced by your own run, with seeds, assumptions, and mean ± standard deviation.